# 04: Network Analysis

This notebook handles:
- Building protein interaction networks using NetworkX
- Identifying network hubs (highly connected proteins)
- Community detection and clustering
- Testing if high pLLPS proteins are network hubs

**Inputs:**
- `results/string_interactions_matched.csv` - From notebook 02
- `results/membrane_proteins.csv` - From notebook 01

**Outputs:**
- `results/hub_analysis_results.csv` - Hub proteins and metrics
- `results/community_analysis.csv` - Community structure
- `results/network_stats.json` - Network statistics

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import llps_functions as lf
import networkx as nx
from collections import Counter

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Previous Results

In [ ]:
# Load data
matched_df = lf.load_analysis_result('string_interactions_matched', format='csv')
membrane_df = lf.load_analysis_result('membrane_proteins', format='csv')

print(f"\n📊 Loaded data:")
print(f"   Matched interactions: {len(matched_df)}")
print(f"   Membrane proteins: {len(membrane_df)}")

## 2. Build Network with NetworkX

Create a network graph from the protein interactions.

In [ ]:
# Use analyze_network function from llps_functions
print("\n🔨 Building network with NetworkX...")
print("   This may take a moment for large networks...")

# Filter to complete pairs only
complete_pairs = matched_df[matched_df['both_in_dataset']].copy()
print(f"   Using {len(complete_pairs)} complete interaction pairs")

# Run network analysis
network_results = lf.analyze_network(
    complete_pairs,
    pllps_threshold=0.7,
    top_n_hubs=20
)

if network_results:
    print(f"\n✅ Network analysis complete")
    print(f"   Network nodes: {network_results['network_stats']['num_nodes']}")
    print(f"   Network edges: {network_results['network_stats']['num_edges']}")
    print(f"   Average degree: {network_results['network_stats']['avg_degree']:.2f}")
    print(f"   Network density: {network_results['network_stats']['density']:.4f}")
else:
    print("\n⚠️  Could not perform network analysis")

## 3. Hub Analysis

Identify network hubs and test if high pLLPS proteins are hubs.

In [ ]:
# Display hub proteins
if network_results and network_results['hubs_df'] is not None:
    hubs_df = network_results['hubs_df']
    
    print(f"\n🌟 Top Network Hubs (Top {len(hubs_df)}):")
    display(hubs_df.head(20))
    
    # Statistics on hubs
    print(f"\n📊 Hub Statistics:")
    print(f"   High pLLPS hubs: {(hubs_df['is_high_pllps'] == True).sum()}")
    print(f"   Percentage high pLLPS: {(hubs_df['is_high_pllps'] == True).sum() / len(hubs_df) * 100:.1f}%")
    print(f"   Average pLLPS of hubs: {hubs_df['pllps_score'].mean():.3f}")
    print(f"   Average degree of hubs: {hubs_df['degree'].mean():.1f}")

In [ ]:
# Test enrichment
if network_results and network_results['hub_enrichment'] is not None:
    enrichment = network_results['hub_enrichment']
    
    print(f"\n🔬 Hub Enrichment Test:")
    print(f"   High pLLPS proteins as hubs: {enrichment['high_pllps_hubs']}")
    print(f"   Expected: {enrichment['expected_high_pllps_hubs']:.1f}")
    print(f"   Enrichment: {enrichment['enrichment']:.2f}x")
    print(f"   P-value: {enrichment['p_value']:.4e}")
    
    if enrichment['p_value'] < 0.05:
        print(f"\n   ✅ Significant enrichment of high pLLPS in hubs (p < 0.05)")
    else:
        print(f"\n   ⚠️  No significant enrichment (p >= 0.05)")

In [ ]:
# Visualize hub characteristics
if network_results and network_results['hubs_df'] is not None:
    hubs_df = network_results['hubs_df']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Degree distribution
    axes[0].hist(hubs_df['degree'], bins=20, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Degree (Number of Connections)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Hub Degree Distribution')
    
    # pLLPS score distribution of hubs
    high_hubs = hubs_df[hubs_df['is_high_pllps']]
    not_high_hubs = hubs_df[~hubs_df['is_high_pllps']]
    
    axes[1].hist([high_hubs['pllps_score'].dropna(), not_high_hubs['pllps_score'].dropna()], 
                bins=20, label=['High pLLPS', 'Not High pLLPS'], alpha=0.7, edgecolor='black')
    axes[1].set_xlabel('pLLPS Score')
    axes[1].set_ylabel('Count')
    axes[1].set_title('pLLPS Distribution of Hub Proteins')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('results/hub_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Saved plot: results/hub_analysis.png")

## 4. Community Detection

Identify communities/clusters in the network.

In [ ]:
# Community analysis
if network_results and network_results['communities'] is not None:
    communities = network_results['communities']
    
    print(f"\n🔍 Community Detection Results:")
    print(f"   Number of communities: {len(communities)}")
    
    # Community size distribution
    community_sizes = [len(comm) for comm in communities]
    print(f"   Largest community: {max(community_sizes)} proteins")
    print(f"   Smallest community: {min(community_sizes)} proteins")
    print(f"   Average community size: {np.mean(community_sizes):.1f}")
    
    # Create community DataFrame
    community_data = []
    for i, comm in enumerate(communities):
        # Get pLLPS scores for community members
        comm_list = list(comm)
        comm_pllps = complete_pairs[
            complete_pairs['protein1'].isin(comm_list) | 
            complete_pairs['protein2'].isin(comm_list)
        ]
        
        # Count high pLLPS members
        pllps_scores = []
        for protein in comm_list:
            scores = complete_pairs[
                (complete_pairs['protein1'] == protein) | 
                (complete_pairs['protein2'] == protein)
            ]
            if len(scores) > 0:
                score = scores.iloc[0]['pllps_1'] if scores.iloc[0]['protein1'] == protein else scores.iloc[0]['pllps_2']
                if pd.notna(score):
                    pllps_scores.append(score)
        
        if pllps_scores:
            community_data.append({
                'community_id': i,
                'size': len(comm),
                'avg_pllps': np.mean(pllps_scores),
                'high_pllps_count': sum(1 for s in pllps_scores if s >= 0.7),
                'high_pllps_fraction': sum(1 for s in pllps_scores if s >= 0.7) / len(pllps_scores)
            })
    
    community_df = pd.DataFrame(community_data)
    
    print("\n📋 Top Communities by Size:")
    display(community_df.sort_values('size', ascending=False).head(10))
else:
    print("\n⚠️  Community detection not available")
    community_df = None

In [ ]:
# Visualize community characteristics
if community_df is not None and len(community_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Community size distribution
    axes[0].hist(community_df['size'], bins=20, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Community Size')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Community Size Distribution')
    
    # Average pLLPS vs community size
    scatter = axes[1].scatter(community_df['size'], community_df['avg_pllps'], 
                              c=community_df['high_pllps_fraction'], cmap='RdYlGn',
                              s=100, alpha=0.7, edgecolors='black')
    axes[1].set_xlabel('Community Size')
    axes[1].set_ylabel('Average pLLPS')
    axes[1].set_title('Community pLLPS vs Size')
    cbar = plt.colorbar(scatter, ax=axes[1])
    cbar.set_label('Fraction High pLLPS')
    
    plt.tight_layout()
    plt.savefig('results/community_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Saved plot: results/community_analysis.png")

## 5. Save Results

In [ ]:
# Save hub analysis
if network_results and network_results['hubs_df'] is not None:
    lf.save_analysis_result(network_results['hubs_df'], 'hub_analysis_results', format='csv')

# Save community analysis
if community_df is not None:
    lf.save_analysis_result(community_df, 'community_analysis_results', format='csv')

# Save network statistics
if network_results:
    stats = {
        'network_stats': network_results['network_stats'],
        'hub_enrichment': network_results['hub_enrichment'] if network_results['hub_enrichment'] else {},
        'num_communities': len(network_results['communities']) if network_results['communities'] else 0
    }
    lf.save_analysis_result(stats, 'network_stats', format='json')

print("\n" + "="*60)
print("✅ All results saved successfully!")
print("="*60)

In [ ]:
# List saved files
lf.list_saved_results()

## Summary

✅ **Completed:**
1. Built protein interaction network using NetworkX
2. Identified network hubs (highly connected proteins)
3. Tested if high pLLPS proteins are enriched in hubs
4. Performed community detection and clustering
5. Analyzed community characteristics
6. Saved all results to `results/` directory

**Next step:** Run `05_functional_groups.ipynb` for functional category analysis.